# Exploratory Data Analysis (EDA)
## Occupancy Prediction for Antalya/Alanya Resort Hotels Using Google Trends

**Prepared by:** Bedirhan Sar  
**Project:** Hotel Occupancy Analysis and Prediction

---

## Executive Summary

This study evaluates whether Google Trends can provide useful external signal for understanding and later predicting **daily hotel occupancy rates** for two resort hotels in the Antalya/Alanya region of Turkey.

The analysis combines:
- hotel occupancy data from **Side Mare Hotel** and **Azura Deluxe**,
- Google Trends data from **Germany, the Netherlands, the United Kingdom, and Türkiye**.

The main result of the exploratory phase is that **same-day Google Trends relationships are generally weak to moderate**, while **several lagged Google Trends features show stronger associations with occupancy**. This suggests that search behavior is more useful as an **early signal** than as a same-day explanatory variable.


## Project Motivation

Resort hotel demand is shaped by strong seasonality, travel planning cycles, destination interest, and B2B distribution channels such as agencies and tour operators. Because of this market structure, Google Trends should not be treated as a direct measure of bookings. Instead, it is better interpreted as a possible upstream indicator of travel intent.

The main research question is:

**Can Google Trends data provide useful information about future occupancy patterns for Antalya/Alanya resort hotels?**


## Data Sources

### Hotel Data
Daily occupancy data was prepared for:
- **Side Mare Hotel**
- **Azura Deluxe**

Primary target variable:
- `occupancy_rate`

### Google Trends Data
Tourism-related Google Trends data was collected for:
- **Germany**
- **Netherlands**
- **United Kingdom**
- **Türkiye**

The raw Trends data was initially organized around:
- date,
- country,
- keyword,
- trend score.


## Data Preparation Workflow

The analytical dataset was built in four steps:

1. **Hotel standardization**  
   Common schema:
   - `date`
   - `hotel_name`
   - `occupancy_rate`

2. **Google Trends standardization**  
   Common schema:
   - `date`
   - `country`
   - `keyword`
   - `google_trends_score`

3. **Pivoting to wide format**  
   Trend variables were converted into columns such as:
   - `trends_germany_alanya_hotel`
   - `trends_turkiye_side_otel`
   - `trends_netherlands_hotel_alanya`

4. **Building the master table**  
   The hotel data and trend data were merged by `date`, producing a dataset where:

   **one row = one hotel on one date**


## Master Table Overview

| Metric | Value |
|---|---:|
| Total rows | 1307 |
| Total columns | 19 |
| Hotels | 2 |
| Date range | 2023-03-25 to 2025-11-08 |
| Duplicate `(date, hotel_name)` rows | 0 |
| Trends missing values per feature | 78 |


In [ ]:
import pandas as pd

master = pd.read_excel("hotel_master_table.xlsx", sheet_name="master_table")
master["date"] = pd.to_datetime(master["date"])
master.head()


## Data Quality Assessment

The merged dataset was reviewed for:
- data type consistency,
- duplicate rows,
- missing values,
- and date-range overlap.

### Summary assessment
The dataset was suitable for analysis. No duplicate `(date, hotel_name)` rows were found. Missing values were concentrated in the Trends variables and were mainly caused by partial mismatch between hotel coverage and Trends collection windows.


In [ ]:
master.info()
master.isna().sum().sort_values(ascending=False).head(20)


## Research Hypotheses

### H1
Google Trends features have a measurable relationship with hotel occupancy.

### H2
Lagged Google Trends values are more informative than same-day Google Trends values.

### H3
The strength of the relationship varies across country-keyword combinations.

### H4
Occupancy exhibits strong seasonal structure, which must be taken into account when interpreting external signals.


## Occupancy Summary by Hotel

| Hotel | Count | Mean | Median | Std. Dev. | Min | Max |
|---|---:|---:|---:|---:|---:|---:|
| Azura Deluxe | 694 | 82.5533 | 94.385 | 23.0493 | 0.00 | 103.56 |
| Side Mare Hotel | 613 | 93.2150 | 97.813 | 14.9361 | 0.00 | 103.75 |

### Interpretation
- Side Mare Hotel has the higher average occupancy level.
- Azura Deluxe shows larger variability.
- Both hotels display strong seasonality.
- Near-zero values are present in low-demand or likely off-season periods.


In [ ]:
master.groupby("hotel_name")["occupancy_rate"].agg(["count", "mean", "median", "std", "min", "max"])


## Visualization 1: Daily Occupancy Over Time

![Daily Occupancy Over Time](Visualizations/occupancy_over_time.png)

This figure highlights the strong seasonal structure of the data and shows that the two hotels follow broadly similar annual rhythms, while still displaying different levels of volatility.


## Visualization 2: Monthly Average Occupancy

![Monthly Average Occupancy](Visualizations/monthly_occupancy.png)

The monthly view makes the seasonality easier to read by smoothing daily noise. The repeated annual pattern confirms that occupancy is strongly seasonal in both hotels.


## Same-Day Correlation Analysis

The first statistical comparison measured whether same-day Google Trends values were associated with same-day occupancy.

### Top Same-Day Pearson Correlations

| Feature | Pearson r | p-value |
|---|---:|---:|
| trends_turkiye_alanya_otel | 0.282470 | 5.58934e-24 |
| trends_turkiye_side_otel | 0.249920 | 5.89086e-19 |
| trends_germany_alanya_hotel | 0.239346 | 1.79783e-17 |
| trends_turkiye_alanya_otel_fiyatlari | 0.219890 | 6.36611e-15 |
| trends_netherlands_hotel_alanya | 0.148717 | 1.62815e-07 |

![Top Same-Day Correlations](Visualizations/top_same_day_correlations.png)

### Interpretation
The same-day relationships are generally weak to moderate. This is consistent with the market structure of the hotels: occupancy is influenced by booking pipelines and agency-driven flows, so same-day search activity is not expected to fully explain same-day stays.


In [ ]:
from scipy.stats import pearsonr, spearmanr

trend_cols = [c for c in master.columns if c.startswith("trends_")]

rows = []
for c in trend_cols:
    sub = master[["occupancy_rate", c]].dropna()
    if len(sub) > 10 and sub[c].nunique() > 1:
        p = pearsonr(sub["occupancy_rate"], sub[c])
        s = spearmanr(sub["occupancy_rate"], sub[c], nan_policy="omit")
        rows.append({
            "feature": c,
            "pearson_r": p.statistic,
            "pearson_p": p.pvalue,
            "spearman_rho": s.statistic,
            "spearman_p": s.pvalue,
        })

same_day_corr = pd.DataFrame(rows).sort_values("pearson_r", ascending=False)
same_day_corr.head(10)


## Lag Analysis

Because travel-related search behavior may occur days or weeks before an actual hotel stay, lagged Trends features were examined at 7, 14, 21, and 28 days.

### Top Best Lagged Pearson Correlations

| Feature | Best Lag (days) | Pearson r | p-value |
|---|---:|---:|---:|
| trends_turkiye_side_otel | 28 | 0.399829 | 1.16762e-44 |
| trends_germany_alanya_hotel | 7 | 0.255002 | 2.51628e-19 |
| trends_turkiye_alanya_otel_fiyatlari | 21 | 0.229107 | 3.47173e-15 |
| trends_netherlands_hotel_alanya | 28 | 0.207813 | 1.68567e-12 |
| trends_turkiye_antalya_tatil | 28 | 0.191652 | 8.11092e-11 |

![Top Lagged Correlations](Visualizations/top_lagged_correlations.png)

### Interpretation
The lag analysis provides one of the most important results of the exploratory phase: several Trends variables become more informative when shifted backward in time. This supports the idea that search interest is more relevant as an **early indicator** than as an immediate signal.


In [ ]:
trend_df = master[["date"] + trend_cols].drop_duplicates("date").sort_values("date").reset_index(drop=True)

lag_rows = []
for lag in [7, 14, 21, 28]:
    lagged = trend_df.copy()
    lagged[trend_cols] = lagged[trend_cols].shift(lag)
    merged = master[["date", "hotel_name", "occupancy_rate"]].merge(lagged, on="date", how="left")
    for c in trend_cols:
        sub = merged[["occupancy_rate", c]].dropna()
        if len(sub) > 10 and sub[c].nunique() > 1:
            p = pearsonr(sub["occupancy_rate"], sub[c])
            lag_rows.append({
                "feature": c,
                "lag_days": lag,
                "pearson_r": p.statistic,
                "pearson_p": p.pvalue,
            })

lag_corr = pd.DataFrame(lag_rows)
best_by_feature = lag_corr.loc[lag_corr.groupby("feature")["pearson_r"].idxmax()].sort_values("pearson_r", ascending=False)
best_by_feature.head(10)


## Visualization 3: Best Lagged Signal Overlay

![Best Lag Overlay](Visualizations/best_lag_overlay.png)

This overlay compares normalized occupancy with the strongest lagged Google Trends signal. It does not prove causality, but it provides visual support for the idea that delayed search behavior may move in a pattern that is more aligned with occupancy than same-day search behavior.


## Formal Hypothesis Assessment

| Hypothesis | Assessment | Summary |
|---|---|---|
| H1: Google Trends features have a measurable relationship with hotel occupancy. | **Partially supported** | Some country-keyword combinations show meaningful association, but the effect is not uniformly strong across all variables. |
| H2: Lagged Google Trends values are more informative than same-day values. | **Supported** | Several lagged features exhibit stronger correlations than their same-day counterparts. |
| H3: Relationship strength varies by country and keyword. | **Supported** | Search terms from Türkiye and Germany appear stronger than many of the United Kingdom features. |
| H4: Occupancy has strong seasonal structure. | **Supported** | Occupancy displays a clear seasonal pattern in both daily and monthly visualizations. |


## Key Analytical Conclusions

1. **Seasonality is the dominant structural pattern** in the occupancy data.
2. **Same-day Google Trends effects are limited**, and should not be overstated.
3. **Lagged Google Trends signals are more promising**, especially for selected country-keyword pairs.
4. **Not all search features are equally useful**, so feature selection is necessary before modeling.
5. Google Trends should be interpreted as a **supporting explanatory layer**, not as a complete representation of hotel demand.


## Hotel-wise Normalization Robustness Check

Because this project pools **two different hotels**,regarding the recommendation of my LA, an additional robustness step was added before making stronger pooled conclusions. The concern is that some raw pooled relationships may partly reflect **between-hotel level differences** rather than genuine **within-hotel movement**.

To address this, occupancy was normalized **within each hotel** using a hotel-wise z-score:

`occupancy_rate_hotel_z = (occupancy_rate - hotel_mean) / hotel_std`

This robustness check does **not replace** the original EDA results. Instead, it tests whether the main same-day and lagged Google Trends findings remain broadly similar after removing hotel-specific occupancy scale differences.

The main question of this section is:

> Do the strongest pooled Google Trends signals remain visible after controlling for hotel-level occupancy differences?

If the ranking and strength of the leading signals remain similar, then the earlier pooled EDA conclusions become more defensible.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import pearsonr

viz_dir = Path('Visualizations')
viz_dir.mkdir(exist_ok=True)

# Hotel-wise normalization for pooled robustness analysis.
hotel_stats = master.groupby('hotel_name')['occupancy_rate'].agg(['mean', 'std']).rename(columns={'mean': 'hotel_mean', 'std': 'hotel_std'})
master_norm = master.merge(hotel_stats, left_on='hotel_name', right_index=True, how='left').copy()
master_norm['occupancy_rate_hotel_z'] = (master_norm['occupancy_rate'] - master_norm['hotel_mean']) / master_norm['hotel_std']

def safe_pearson(x, y):
    sub = pd.concat([x, y], axis=1).dropna()
    if len(sub) <= 10 or sub.iloc[:, 0].nunique() <= 1 or sub.iloc[:, 1].nunique() <= 1:
        return np.nan, np.nan
    p = pearsonr(sub.iloc[:, 0], sub.iloc[:, 1])
    return p.statistic, p.pvalue

# Same-day raw vs hotel-normalized correlations.
same_day_norm_rows = []
for c in trend_cols:
    r_raw, p_raw = safe_pearson(master_norm[c], master_norm['occupancy_rate'])
    r_norm, p_norm = safe_pearson(master_norm[c], master_norm['occupancy_rate_hotel_z'])
    same_day_norm_rows.append({
        'feature': c,
        'pearson_raw': r_raw,
        'pearson_p_raw': p_raw,
        'pearson_hotel_z': r_norm,
        'pearson_p_hotel_z': p_norm,
        'abs_change': abs(r_norm) - abs(r_raw) if pd.notna(r_raw) and pd.notna(r_norm) else np.nan,
    })
same_day_norm = pd.DataFrame(same_day_norm_rows).sort_values('pearson_hotel_z', ascending=False)

# Lagged raw vs hotel-normalized correlations.
lag_norm_rows = []
trend_unique = master_norm[['date'] + trend_cols].drop_duplicates('date').sort_values('date').reset_index(drop=True)
for lag in [7, 14, 21, 28]:
    lagged = trend_unique.copy()
    lagged[trend_cols] = lagged[trend_cols].shift(lag)
    merged = master_norm[['date', 'hotel_name', 'occupancy_rate', 'occupancy_rate_hotel_z']].merge(lagged, on='date', how='left')
    for c in trend_cols:
        r_raw, p_raw = safe_pearson(merged[c], merged['occupancy_rate'])
        r_norm, p_norm = safe_pearson(merged[c], merged['occupancy_rate_hotel_z'])
        lag_norm_rows.append({
            'feature': c,
            'lag_days': lag,
            'pearson_raw': r_raw,
            'pearson_p_raw': p_raw,
            'pearson_hotel_z': r_norm,
            'pearson_p_hotel_z': p_norm,
            'abs_change': abs(r_norm) - abs(r_raw) if pd.notna(r_raw) and pd.notna(r_norm) else np.nan,
        })
lag_norm = pd.DataFrame(lag_norm_rows)
best_lag_norm = lag_norm.loc[lag_norm.groupby('feature')['pearson_hotel_z'].idxmax()].sort_values('pearson_hotel_z', ascending=False)

# Same-day comparison plot.
same_plot = same_day_norm.nlargest(8, 'pearson_hotel_z').sort_values('pearson_hotel_z')
plt.figure(figsize=(10, 5))
plt.barh(same_plot['feature'], same_plot['pearson_raw'], alpha=0.7, label='Raw occupancy')
plt.barh(same_plot['feature'], same_plot['pearson_hotel_z'], alpha=0.7, label='Hotel-wise normalized occupancy')
plt.xlabel('Pearson correlation')
plt.title('Same-day correlations: raw vs hotel-wise normalized occupancy')
plt.legend()
plt.tight_layout()
plt.savefig(viz_dir / 'hotelwise_normalization_same_day_compare.png', dpi=150)
plt.show()

# Lagged comparison plot.
lag_plot = best_lag_norm.nlargest(8, 'pearson_hotel_z').sort_values('pearson_hotel_z')
plt.figure(figsize=(10, 5))
plt.barh(lag_plot['feature'], lag_plot['pearson_raw'], alpha=0.7, label='Raw occupancy')
plt.barh(lag_plot['feature'], lag_plot['pearson_hotel_z'], alpha=0.7, label='Hotel-wise normalized occupancy')
plt.xlabel('Best Pearson correlation across tested lags')
plt.title('Best lagged correlations: raw vs hotel-wise normalized occupancy')
plt.legend()
plt.tight_layout()
plt.savefig(viz_dir / 'hotelwise_normalization_lagged_compare.png', dpi=150)
plt.show()

same_day_norm.head(10), best_lag_norm.head(10)


### Visualization 4: Same-Day Robustness Comparison (Raw vs Hotel-wise Normalized Occupancy)

![Hotel-wise normalization same-day comparison](Visualizations/hotelwise_normalization_same_day_compare.png)

This figure compares the strongest same-day Pearson correlations under two targets: the original raw `occupancy_rate` and the hotel-wise normalized `occupancy_rate_hotel_z`. The goal is to test whether the same-day ranking is stable after removing hotel-specific average occupancy differences.

### Visualization 5: Lagged Robustness Comparison (Best Lag per Feature, Raw vs Hotel-wise Normalized Occupancy)

![Hotel-wise normalization lagged comparison](Visualizations/hotelwise_normalization_lagged_compare.png)

This figure compares the strongest lagged Google Trends signals under the same two targets. This is the more important robustness figure because the main EDA result already showed that **lagged Trends is more informative than same-day Trends**.


### Robustness Findings

The hotel-wise normalization results show that the **main ranking is broadly preserved** after controlling for cross-hotel occupancy scale differences. This means the earlier pooled findings are **not just an artifact of combining two hotels with different average occupancy levels**.

#### Top Same-Day Features: Raw vs Hotel-wise Normalized Occupancy

| Feature | Pearson r (raw occupancy) | Pearson r (hotel-wise normalized occupancy) |
|---|---:|---:|
| trends_turkiye_alanya_otel | 0.282470 | 0.260386 |
| trends_germany_alanya_hotel | 0.239346 | 0.240763 |
| trends_turkiye_side_otel | 0.249920 | 0.234412 |
| trends_turkiye_alanya_otel_fiyatlari | 0.219890 | 0.214309 |
| trends_netherlands_hotel_alanya | 0.148717 | 0.146934 |

#### Top Lagged Features: Raw vs Hotel-wise Normalized Occupancy

| Feature | Best lag (days) | Pearson r (raw occupancy) | Pearson r (hotel-wise normalized occupancy) |
|---|---:|---:|---:|
| trends_turkiye_side_otel | 28 | 0.399829 | 0.401542 |
| trends_germany_alanya_hotel | 28 | 0.251241 | 0.257707 |
| trends_turkiye_alanya_otel_fiyatlari | 14 | 0.226285 | 0.226418 |
| trends_netherlands_hotel_alanya | 28 | 0.207813 | 0.205630 |
| trends_turkiye_antalya_tatil | 28 | 0.191652 | 0.191074 |

#### Interpretation

Several important conclusions remain unchanged after hotel-wise normalization:

- **Lagged Google Trends remains stronger than same-day Google Trends.**
- The strongest overall signal remains **`trends_turkiye_side_otel` at 28 days**.
- The most useful signals still come mainly from **Türkiye** and **Germany**.
- Same-day relationships remain weaker and should still be interpreted cautiously.

The strongest robustness result is that **`trends_turkiye_side_otel` stays the leading feature even after normalization**, and its lagged correlation is slightly stronger under the normalized target than under the raw target. This strengthens the earlier interpretation that Google Trends is more useful as an **early travel-intent signal** than as a same-day demand proxy.

Overall, this robustness layer supports the earlier pooled EDA conclusions rather than overturning them.

### Implication for ML Stage

This robustness check should be reflected in the ML stage as an additional **robustness specification**, not as a replacement of the main target variable.

- The main prediction target should remain **raw `occupancy_rate`**, because it is the real business outcome.
- Hotel-wise normalized occupancy should be used as a **secondary robustness target**.
- In the next modeling stage, it is useful to check whether the same lagged Google Trends features remain important under both targets.

If the same lagged features remain useful in both setups, then the modeling conclusions become more defensible.


## Limitations

1. **Google Trends is not booking data**  
   It measures search interest, not confirmed reservations.

2. **B2B demand channels are important**  
   Tour operators and agencies likely explain part of occupancy behavior that search data does not capture.

3. **Correlation does not establish causation**  
   These relationships identify association, not direct causal mechanisms.

4. **Seasonality can inflate apparent relationships**  
   Since both Trends and occupancy may rise during the same periods, part of the observed relationship may reflect shared seasonality.

5. **Feature quality differs across keywords and countries**  
   Some search features are informative, while others may mostly contribute noise.


## Next Step

The next stage is to convert the most promising Trends variables into lagged model features and combine them with:
- calendar information,
- hotel identity,
- and past occupancy values.

That modeling stage will test whether the exploratory relationships observed here translate into measurable predictive improvement.


## Final Summary

This exploratory analysis established a structured foundation for the predictive phase of the project.

In summary:
- two hotel occupancy datasets were cleaned and combined,
- four-country Google Trends data was collected and integrated,
- a master analytical table was built,
- data quality checks were completed,
- seasonal occupancy structure was documented,
- same-day and lagged statistical relationships were tested,
- and lagged search features emerged as the most promising external signals.

Overall, the evidence suggests that Google Trends is not sufficient on its own to explain resort hotel occupancy, but selected lagged search features may provide useful incremental information within a broader forecasting framework.
